# Composite ranking: join LRT + delta_psi + rho_W per lineage

Same procedure as the mouse notebook `analysis/mouse_svz/06_260411-optimized2_composite_ranking.ipynb`. Reads the per-lineage outputs of `09_260504-effect_sizes_per_lineage.ipynb` and joins them onto the DirMult LRT result, recomputes BH-adjusted p-values on the converged subset, and computes the composite score:

`composite_score_abs = 0.50 * scaled_neglog10_padj + 0.25 * rank(|delta_psi|) + 0.25 * rank(|rho_W|)`

`composite_score_signed = sign(delta_psi) * composite_score_abs`

Also classifies each gene's trajectory (monotonic_lengthening, monotonic_shortening, transient, mixed) using the Δψ + NMI + t_at_max/min from the delta_psi output.

Output: `composite_ranking/composite_ranking_optimized2_<lineage>.csv` per lineage.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.size": 7,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "axes.titlesize": 7,
        "axes.labelsize": 7,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
    }
)
print("Setup complete.")

In [ ]:
LINEAGE_TAG = {"Inhibitory neurons": "inhibitory", "Excitatory neurons": "excitatory"}
W_PADJ, W_DPSI, W_RHO = 0.50, 0.25, 0.25

# Trajectory classification thresholds (t_at_max / t_at_min are pseudotime values
# in [q_early, q_late] = [0.1, 0.9] of the lineage population).
NMI_MONO_THR = 0.20  # NMI <= 0.20 -> "monotonic"
NMI_TRANSIENT_THR = 0.50  # NMI >  0.50 -> "transient"
T_BOUNDARY = 0.20  # 20% of the [q_early, q_late] range from each end

BASEPATH = Path(f"{DATA_ROOT}/3utr/organoids/knoblich")
UTR_DIR = BASEPATH / "utrcalling"
DM_RESULTS_DIR = UTR_DIR / "dirmult_results"
EFFECT_SIZES_DIR = UTR_DIR / "effect_sizes"
OUT_DIR = UTR_DIR / "composite_ranking"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASEPATH:        {BASEPATH}")
print(f"DirMult results: {DM_RESULTS_DIR}")
print(f"Effect sizes:    {EFFECT_SIZES_DIR}")
print(f"Output:          {OUT_DIR}")

## Per-lineage join and composite score


In [ ]:
def build_composite_ranking(lineage_tag):
    print(f"\n=== {lineage_tag} ===")

    # Inputs
    dm_csv = (
        DM_RESULTS_DIR
        / lineage_tag
        / "controls_only_batch_number"
        / f"dirmult_regression_optimized2_joint_df5_pt-dpt_pseudotime_{lineage_tag}_cov-batch_number.csv"
    )
    dpsi_csv = EFFECT_SIZES_DIR / lineage_tag / "delta_psi_per_gene_df5_q0.1-0.9.csv"
    rho_csv = EFFECT_SIZES_DIR / lineage_tag / "rho_W_per_gene.csv"

    for p in [dm_csv, dpsi_csv, rho_csv]:
        assert p.exists(), f"missing input: {p}"

    # Load
    dm = pd.read_csv(dm_csv)
    dpsi = pd.read_csv(dpsi_csv)
    rho = pd.read_csv(rho_csv)
    print(f"  DirMult: {dm.shape}")
    print(f"  delta_psi: {dpsi.shape}")
    print(f"  rho_W: {rho.shape}")

    # Standardise key columns
    dm = dm.rename(columns={"utr_name": "gene_id", "padj": "padj_mixed"})

    # Recompute padj on converged subset (as in the mouse notebook)
    dm["padj_chisq"] = np.nan
    converged_mask = (dm["converged"] == True) & (dm["lrt_status"] == "ok")
    n_total = len(dm)
    n_converged = int(converged_mask.sum())
    n_nonconv = n_total - n_converged
    print(
        f"  Total genes: {n_total:,}; converged: {n_converged:,}; non-converged: {n_nonconv:,}"
    )
    if n_converged > 0:
        conv_idx = dm.index[converged_mask]
        _, padj_chisq, _, _ = multipletests(
            dm.loc[conv_idx, "p_value_chisq"].values, method="fdr_bh"
        )
        dm.loc[conv_idx, "padj_chisq"] = padj_chisq
    dm["padj"] = dm["padj_chisq"]

    # Bootstrap padj if available (empty when run with --bootstrap 0)
    dm["padj_boot"] = np.nan
    boot_mask = dm["p_value_boot"].notna() & converged_mask
    if boot_mask.sum() > 0:
        boot_idx = dm.index[boot_mask]
        _, padj_boot, _, _ = multipletests(
            dm.loc[boot_idx, "p_value_boot"].values, method="fdr_bh"
        )
        dm.loc[boot_idx, "padj_boot"] = padj_boot

    print(f"  Discoveries (padj_chisq < 0.05): {(dm['padj_chisq'] < 0.05).sum()}")

    # Inner join: DirMult-converged + delta_psi + rho_W
    dm_conv = dm[converged_mask].copy()
    df = dm_conv.merge(dpsi, on="gene_id", suffixes=("_dm", "_dp"))
    print(f"  After join with delta_psi: {len(df):,}")
    # Note: rho_W has gene_name from harmony h5ad; drop dpsi's redundant gene_name
    if "gene_name_dp" in df.columns:
        df = df.drop(columns=["gene_name_dp"]).rename(
            columns={"gene_name_dm": "gene_name"}
        )
    elif "gene_name" not in df.columns and "gene_name_dm" in df.columns:
        df = df.rename(columns={"gene_name_dm": "gene_name"})
    df = df.merge(
        rho[["gene_id", "rho_W", "abs_rho_W", "padj_BH"]].rename(
            columns={"padj_BH": "padj_BH_rho_W"}
        ),
        on="gene_id",
        how="left",
    )
    print(f"  After join with rho_W: {len(df):,}")

    # Filter: must have non-NaN delta_psi and rho_W to compute composite
    n_pre = len(df)
    df = df[df["delta_psi"].notna() & df["rho_W"].notna()].copy()
    print(f"  After non-NaN delta_psi+rho_W: {len(df):,} (dropped {n_pre - len(df)})")

    # Composite score
    neglog10 = -np.log10(df["padj"].clip(lower=1e-300))
    df["scaled_neglog10_padj"] = (neglog10 - neglog10.min()) / (
        neglog10.max() - neglog10.min()
    )
    df["rank_delta_psi"] = df["delta_psi"].abs().rank(pct=True)
    df["rank_abs_rho_W"] = df["abs_rho_W"].rank(pct=True)
    df["composite_score_abs"] = (
        W_PADJ * df["scaled_neglog10_padj"]
        + W_DPSI * df["rank_delta_psi"]
        + W_RHO * df["rank_abs_rho_W"]
    )
    df["composite_score_signed"] = np.sign(df["delta_psi"]) * df["composite_score_abs"]
    df = df.sort_values("composite_score_abs", ascending=False).reset_index(drop=True)

    # Trajectory classification
    # t_at_max / t_at_min are absolute pseudotime values; compute relative position
    # in the [q_early=0.1, q_late=0.9] range.
    pt_min = df["t_at_min"].min()
    pt_max = df["t_at_max"].max()
    pt_range = pt_max - pt_min
    if pt_range > 0:
        df["t_at_max_rel"] = (df["t_at_max"] - pt_min) / pt_range
        df["t_at_min_rel"] = (df["t_at_min"] - pt_min) / pt_range
    else:
        df["t_at_max_rel"] = np.nan
        df["t_at_min_rel"] = np.nan

    def classify(row):
        nmi = row["nmi"]
        dpsi_v = row["delta_psi"]
        if np.isnan(nmi) or np.isnan(dpsi_v):
            return "unclassified"
        if nmi < NMI_MONO_THR:
            return "monotonic_lengthening" if dpsi_v > 0 else "monotonic_shortening"
        if nmi > NMI_TRANSIENT_THR:
            tmax_rel = row.get("t_at_max_rel", 0.5)
            tmin_rel = row.get("t_at_min_rel", 0.5)
            if pd.isna(tmax_rel) or pd.isna(tmin_rel):
                return "transient_mixed"
            late = (tmax_rel > 1 - T_BOUNDARY) or (tmin_rel > 1 - T_BOUNDARY)
            early = (tmax_rel < T_BOUNDARY) or (tmin_rel < T_BOUNDARY)
            if late and not early:
                return "transient_late"
            if early and not late:
                return "transient_early"
            return "transient_mid"
        return "mixed"

    df["trajectory_pattern"] = df.apply(classify, axis=1)

    print(f"  Trajectory pattern counts:")
    print(df["trajectory_pattern"].value_counts().to_string())

    out_path = OUT_DIR / f"composite_ranking_optimized2_{lineage_tag}.csv"
    df.to_csv(out_path, index=False)
    print(f"  Saved {df.shape} -> {out_path}")
    return df


composite_dfs = {}
for ltag in LINEAGE_TAG.values():
    composite_dfs[ltag] = build_composite_ranking(ltag)

## Cross-lineage diagnostics


In [ ]:
inh = composite_dfs["inhibitory"]
exc = composite_dfs["excitatory"]

shared = set(inh["gene_id"]) & set(exc["gene_id"])
print(f"Shared genes (in both lineages): {len(shared):,}")

inh_s = inh.set_index("gene_id").loc[
    list(shared), ["composite_score_abs", "delta_psi", "rho_W"]
]
exc_s = exc.set_index("gene_id").loc[
    list(shared), ["composite_score_abs", "delta_psi", "rho_W"]
]

rho_score, _ = spearmanr(inh_s["composite_score_abs"], exc_s["composite_score_abs"])
rho_dpsi, _ = spearmanr(inh_s["delta_psi"], exc_s["delta_psi"])
rho_rhow, _ = spearmanr(inh_s["rho_W"], exc_s["rho_W"])
print(f"Spearman across lineages (shared genes):")
print(f"  composite_score_abs: rho={rho_score:.3f}")
print(f"  delta_psi:           rho={rho_dpsi:.3f}")
print(f"  rho_W:               rho={rho_rhow:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, col, lo, hi in [
    (axes[0], "composite_score_abs", 0, 1),
    (axes[1], "delta_psi", -1, 1),
    (axes[2], "rho_W", -1, 1),
]:
    ax.scatter(inh_s[col], exc_s[col], s=4, alpha=0.4, rasterized=True, color="black")
    ax.plot([lo, hi], [lo, hi], color="firebrick", linestyle="--", linewidth=0.8)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel(f"inhibitory {col}")
    ax.set_ylabel(f"excitatory {col}")
    ax.set_title(f"{col} (rho={spearmanr(inh_s[col], exc_s[col])[0]:.3f})")

plt.tight_layout()
plt.savefig(OUT_DIR / "cross_lineage_composite_concordance.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Per-lineage histograms
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for row, ltag in enumerate(LINEAGE_TAG.values()):
    df = composite_dfs[ltag]

    axes[row, 0].hist(
        df["delta_psi"].dropna(),
        bins=50,
        color="#2980b9",
        edgecolor="black",
        linewidth=0.3,
    )
    axes[row, 0].axvline(0, color="black", linewidth=0.7, linestyle="--")
    axes[row, 0].set_title(f"{ltag}: delta_psi")
    axes[row, 0].set_xlabel("delta_psi")
    axes[row, 0].set_ylabel("genes")

    axes[row, 1].hist(
        df["rho_W"].dropna(), bins=50, color="#c0392b", edgecolor="black", linewidth=0.3
    )
    axes[row, 1].axvline(0, color="black", linewidth=0.7, linestyle="--")
    axes[row, 1].set_title(f"{ltag}: rho_W")
    axes[row, 1].set_xlabel("rho_W")
    axes[row, 1].set_ylabel("genes")

    axes[row, 2].hist(
        df["composite_score_abs"].dropna(),
        bins=50,
        color="#16a085",
        edgecolor="black",
        linewidth=0.3,
    )
    axes[row, 2].set_title(f"{ltag}: composite_score_abs")
    axes[row, 2].set_xlabel("composite_score_abs")
    axes[row, 2].set_ylabel("genes")

    counts = df["trajectory_pattern"].value_counts()
    axes[row, 3].bar(range(len(counts)), counts.values, color="grey")
    axes[row, 3].set_xticks(range(len(counts)))
    axes[row, 3].set_xticklabels(counts.index, rotation=45, ha="right", fontsize=6)
    axes[row, 3].set_title(f"{ltag}: trajectory_pattern")
    axes[row, 3].set_ylabel("genes")

plt.tight_layout()
plt.savefig(OUT_DIR / "composite_distributions.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Top-10 per lineage by composite_score_abs
for ltag, df in composite_dfs.items():
    print(f"\n=== TOP 10 by composite_score_abs ({ltag}) ===")
    cols = [
        "gene_id",
        "gene_name",
        "K",
        "n_cells_dm",
        "padj",
        "delta_psi",
        "rho_W",
        "nmi",
        "composite_score_abs",
        "composite_score_signed",
        "trajectory_pattern",
    ]
    print(df[cols].head(10).to_string(index=False))